# Detección de fallas en agentes: debugging y análisis de errores

Este notebook usa trazas y métricas para detectar problemas reales en agentes LLM y proponer correcciones.

## Objetivos
- Reconocer fallas comunes en agentes: hallucination, tool misuse, loops y respuestas incompletas.
- Analizar ejecuciones paso a paso.
- Proponer mejoras con un agente corregido.

In [1]:
!pip install pandas langchain langchain-openai wikipedia python-dotenv

  Using cached pandas-3.0.6-cp315-cp315-win_amd64.whl.metadata (19 kB)
  Using cached langchain-1.4.5-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_openai-1.7.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached numpy-2.5.4-cp315-cp315-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached langchain_core-1.6.10-py3-none-any.whl.metadata (4.8 kB)
  Using cached langgraph-1.2.15-py3-none-any.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.35-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.14.8-py3-none-any.whl.metadata (22 kB)
  Using cached pyyaml-6.0.3.tar.gz (130 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished w

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [40 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp315-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\river\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\river\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: it looks like you have an existing rustup settings file at:
      warn: C:\Users\river\AppData\Local\puccinialin\puccinialin\Cache\rustup\settings.toml
      warn: installing msvc toolchain without its prerequisites
      error: could not create link from 'C:\Users\river\AppData\Local\puccinialin\puccinialin\Cache\cargo\bin\rustup.exe' to 'C:\Users\river\AppData\Local\puccinialin\puccinialin\Cache\cargo\bin\cargo-miri.ex

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

if os.environ.get('GITHUB_BASE_URL') and os.environ.get('GITHUB_TOKEN'):
    os.environ['OPENAI_API_BASE'] = os.environ.get('GITHUB_BASE_URL')
    os.environ['OPENAI_API_KEY'] = os.environ.get('GITHUB_TOKEN')

import wikipedia
from langchain.chat_models import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM usando las vars de entorno (si están disponibles)
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0
    )
    print("✅ LLM de LangChain configurado (usando credenciales de .env si existe).")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

ModuleNotFoundError: No module named 'langchain'

In [ ]:
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langchain_classic import hub
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client()
prompt = client.pull_prompt("hormold/openai-functions-agent",
                            dangerously_pull_public_prompt=True)

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

In [ ]:
import random
import time
from pprint import pprint

try:
    import pandas as pd
except ImportError:
    raise ImportError('Instala pandas con `pip install pandas` antes de ejecutar este notebook.')

# Base de conocimiento local para la tarea
knowledge = {
    'agentes': 'Un agente LLM usa herramientas y memoria para completar tareas.',
    'observabilidad': 'La observabilidad captura logs, trazas y métricas.',
    'historia': 'La independencia se celebró en 1810 en muchos países de América Latina.',
}

In [ ]:
class DebugAgent:
    def __init__(self, knowledge):
        self.knowledge = knowledge

    def run(self, question):
        trace = []
        # Paso 1: decidir si usa la herramienta local
        uses_tool = 'buscar' in question.lower() or 'conocimiento' in question.lower()
        trace.append({
            'step': 1,
            'tipo': 'decision',
            'input': question,
            'output': 'usar herramienta local' if uses_tool else 'respuesta directa',
            'latencia': '0.05s',
            'tokens': 10,
            'estado': 'ok'
        })

        if uses_tool:
            found = [value for key, value in self.knowledge.items() if key in question.lower()]
            if found:
                output = found[0]
                trace.append({
                    'step': 2,
                    'tipo': 'tool',
                    'input': question,
                    'output': output,
                    'latencia': '0.10s',
                    'tokens': 0,
                    'estado': 'ok'
                })
            else:
                output = 'No encontré información local, uso conocimiento general.'
                trace.append({
                    'step': 2,
                    'tipo': 'tool',
                    'input': question,
                    'output': 'herramienta fallida',
                    'latencia': '0.10s',
                    'tokens': 0,
                    'estado': 'warning'
                })
                trace.append({
                    'step': 3,
                    'tipo': 'llm',
                    'input': question,
                    'output': 'Respuesta general del modelo.',
                    'latencia': '0.15s',
                    'tokens': 40,
                    'estado': 'ok'
                })
                return trace

        output = 'El agente encontró datos locales y respondió con ellos.'
        trace.append({
            'step': 3,
            'tipo': 'llm',
            'input': question,
            'output': output,
            'latencia': '0.12s',
            'tokens': 35,
            'estado': 'ok'
        })
        return trace

In [ ]:
def analyze_trace(trace):
    issues = []
    for step in trace:
        if step['tipo'] == 'tool' and step['estado'] != 'ok':
            issues.append('Herramienta reportó un problema: ' + step['output'])
        if step['tipo'] == 'llm' and 'general' in step['output'].lower():
            issues.append('Posible hallucination o uso de conocimiento general.')
        if step['step'] > 1 and step['tipo'] == 'llm' and trace[0]['output'] == 'respuesta directa':
            issues.append('El agente respondió directo sin usar herramienta cuando debería haberla usado.')
    return issues

agent = DebugAgent(knowledge)
questions = [
    'Busca en la base de conocimiento local qué es observabilidad.',
    '¿Qué es un agente y cómo usa herramientas?',
    'Consulta datos sobre historia.'
]

for q in questions:
    print(f'\n=== Análisis para pregunta: {q}')
    trace = agent.run(q)
    df = pd.DataFrame(trace)
    display(df)
    issues = analyze_trace(trace)
    print('Issues detectados:', issues)

## Propuesta de corrección

En este caso, podemos mejorar el agente al:
- reforzar la decisión de usar herramienta cuando hay palabras clave relevantes.
- validar si la herramienta encontró resultados antes de pasar al modelo.
- registrar explícitamente cuando se produce una respuesta general.

In [ ]:
class FixedDebugAgent(DebugAgent):
    def run(self, question):
        trace = []
        uses_tool = any(word in question.lower() for word in ['buscar', 'conocimiento', 'consulta', 'datos'])
        trace.append({
            'step': 1,
            'tipo': 'decision',
            'input': question,
            'output': 'usar herramienta local' if uses_tool else 'respuesta directa',
            'latencia': '0.05s',
            'tokens': 10,
            'estado': 'ok'
        })

        if uses_tool:
            found = [value for key, value in self.knowledge.items() if key in question.lower()]
            if found:
                output = found[0]
                trace.append({
                    'step': 2,
                    'tipo': 'tool',
                    'input': question,
                    'output': output,
                    'latencia': '0.10s',
                    'tokens': 0,
                    'estado': 'ok'
                })
                trace.append({
                    'step': 3,
                    'tipo': 'llm',
                    'input': output,
                    'output': f'Usé la información local y respondí: {output}',
                    'latencia': '0.12s',
                    'tokens': 25,
                    'estado': 'ok'
                })
                return trace
            else:
                trace.append({
                    'step': 2,
                    'tipo': 'tool',
                    'input': question,
                    'output': 'no hay datos locales',
                    'latencia': '0.10s',
                    'tokens': 0,
                    'estado': 'warning'
                })
                trace.append({
                    'step': 3,
                    'tipo': 'llm',
                    'input': 'No se encontró información local. Responde con cautela.',
                    'output': 'Informé que no encontré datos locales y no inventé información.',
                    'latencia': '0.15s',
                    'tokens': 45,
                    'estado': 'ok'
                })
                return trace

        trace.append({
            'step': 2,
            'tipo': 'llm',
            'input': question,
            'output': 'Respuesta directa sin necesidad de herramienta.',
            'latencia': '0.12s',
            'tokens': 35,
            'estado': 'ok'
        })
        return trace

# Ejecutar comparación antes/después
fixed_agent = FixedDebugAgent(knowledge)

for q in questions:
    print(f'\n=== Comparación para: {q}')
    trace_before = pd.DataFrame(DebugAgent(knowledge).run(q))
    trace_after = pd.DataFrame(fixed_agent.run(q))
    print('--- Antes ---')
    display(trace_before)
    print('--- Después ---')
    display(trace_after)